# REBULEX — 100 binlik ürün havuzu (Colab)

Ürün havuzu kurulur ve ölçülür:
1. **Çekme:** 10 binlik havuz aynen alınır (test hedefleri içinde), kalan 90 bin her mahkemede **en yeni**
   kararlardan, günümüzden geçmişe doğru (`fetch_sample.py --size 100000`). Kaynak payları aynı.
2. **Temizleme:** `kararlar_temiz.parquet` (orijinal `kararlar.parquet`'e dokunulmaz).
3. **Embedding:** Cosmos (`ytu-ce-cosmos/turkish-e5-large`), tüm karar (per_decision), kısa başlık.
4. **Ölçüm:** v2 ve dilekçe, geniş ölçü 1–100.

Kod GitHub'dan (`metin-temizleme` dalı), veriler Drive'dan gelir.

**Başlamadan önce**
1. `Çalışma zamanı > Çalışma zamanı türünü değiştir` → **A100** ve **Yüksek RAM**.
2. Drive'da `REBULEX/magibu2_veri.zip` olmalı (10k havuzu ve test setleri bunun içinde).
3. Soldaki 🔑 *Secrets* bölümünde `HF_TOKEN` olmalı (notebook erişimi açık). Karar bucket'ı
   (`magibu/turkish-court-decisions-bucket`) private; token'ın hesabı magibu org'unda olmalı.

**Akış:** `Çalışma zamanı > Tümünü çalıştır`. 1. hücrede Drive izni istenir, onayla. Son hücre çalışma zamanını
kapatır. Her adımın çıktısı Drive'da `REBULEX/n100k_sonuc/` altına kopyalanır; oturum koparsa aynı notebook
baştan çalıştırılır, bitmiş adımlar atlanır.

## 1. Drive'ı bağla ve GPU'yu kontrol et

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Ayarlar, kod ve kütüphaneler

In [ ]:
import os

DRIVE_DIR = "/content/drive/MyDrive/REBULEX"
SONUC = f"{DRIVE_DIR}/n100k_sonuc"
LOCAL_DATA = "/content/data"
POOL = f"{LOCAL_DATA}/n100000"
REPO = "https://github.com/ahmetege0/rebulex.git"
BRANCH = "metin-temizleme"

os.environ["REBULEX_DATA"] = LOCAL_DATA        # config.py veriyi buradan okur
os.environ["REBULEX_BATCH_TOKENS"] = "65536"   # L4/A100 için
os.environ["REBULEX_DTYPE"] = "bfloat16"

if not os.path.exists("/content/rebulex"):
    !git clone -q -b {BRANCH} {REPO} /content/rebulex
else:
    !cd /content/rebulex && git fetch -q && git reset -q --hard origin/{BRANCH}
%cd /content/rebulex
!git log --oneline -1
!pip install -q -r requirements.txt

from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("HF_TOKEN yüklendi")

## 3. Verileri hazırla
Zip Colab diskine açılır: 2000 ve 10 bin havuzları, test setleri, `experiments.csv`.
Drive'da önceki bir oturumun çıktıları varsa (100 bin havuzu, temiz metin, vektörler, `experiments.csv`) geri
yüklenir; bitmiş adımlar yeniden çalışmaz.

In [ ]:
import shutil

os.makedirs(SONUC, exist_ok=True)
if not os.path.exists(f"{LOCAL_DATA}/experiments.csv"):   # oturum içinde eklenen satırlar ezilmesin
    !unzip -q -o {DRIVE_DIR}/magibu2_veri.zip -d {LOCAL_DATA}
    if os.path.exists(f"{SONUC}/experiments.csv"):
        shutil.copy(f"{SONUC}/experiments.csv", f"{LOCAL_DATA}/experiments.csv")
        print("Önceki experiments.csv Drive'dan geri yüklendi")
if os.path.exists(f"{SONUC}/n100000"):
    shutil.copytree(f"{SONUC}/n100000", POOL, dirs_exist_ok=True)
    print("Önceki 100 bin dosyaları Drive'dan geri yüklendi")
!find {LOCAL_DATA} -maxdepth 2 | sort


def save():
    """100 bin dosyalarını ve experiments.csv'yi Drive'a kopyalar (vektör veritabanı qdrant hariç)."""
    try:
        shutil.copy(f"{LOCAL_DATA}/experiments.csv", f"{SONUC}/experiments.csv")
        shutil.copytree(POOL, f"{SONUC}/n100000", ignore=shutil.ignore_patterns("qdrant"), dirs_exist_ok=True)
        print("Drive'a kopyalandı")
    except Exception as e:
        print("Drive'a kopyalanamadı:", e)

## 4. 100 bin kararı çek
10 bin havuzu + her kaynakta en yeni kararlar. Yalnızca gereken parquet blokları okunur. Çıktıda her kaynağın
tarih aralığı yazar.

In [ ]:
if not os.path.exists(f"{POOL}/kararlar.parquet"):
    !python -u fetch_sample.py --size 100000 2>&1 | tee -a {SONUC}/gece.log
    save()
else:
    print("100 bin havuzu zaten var, atlanıyor.")

## 5. Temizle
`cleaning.py` her karara uygulanır, `kararlar_temiz.parquet` yazılır.

In [ ]:
if not os.path.exists(f"{POOL}/kararlar_temiz.parquet"):
    !python -u -c "import run_clean_experiments as r; r.make_clean_pool(100000)" 2>&1 | tee -a {SONUC}/gece.log
    save()
else:
    print("Temiz havuz zaten var, atlanıyor.")

## 6. Embedding ve ölçüm
Cosmos, tüm karar (per_decision). Vektörler `n100000/temiz/cosmos_e5/embeddings/` altına yazılır; ürün
veritabanına daha sonra bu dosyadan yüklenecek.

In [ ]:
from datetime import datetime

print(f"Başladı: {datetime.now():%H:%M}")
!python -u run_clean_experiments.py --size 100000 --model cosmos_e5 --deneme per_decision 2>&1 | tee -a {SONUC}/gece.log
save()
print(f"Bitti: {datetime.now():%H:%M}")

## 7. Sonuçlar: Cosmos, tüm karar, temiz metin — 2 bin / 10 bin / 100 bin

In [ ]:
import pandas as pd

try:
    exp = pd.read_csv(f"{LOCAL_DATA}/experiments.csv", encoding="utf-8-sig")
    rows = exp[(exp.model == "ytu-ce-cosmos/turkish-e5-large") & (exp.deneme == "per_decision")
               & exp["not"].astype(str).str.startswith("temiz")]
    cols = ["karar_sayisi", "test_seti", "ilk1_genis_%", "ilk5_genis_%", "ilk10_genis_%", "ilk20_genis_%",
            "ilk100_genis_%", "embedding_sn"]
    pd.set_option("display.width", 250)
    display(rows[cols].sort_values(["test_seti", "karar_sayisi"]))
except Exception as e:
    print("Tablo basılamadı:", e)

## 8. Çalışma zamanını kapat
Boşta birim harcanmasın. Sonuçlar Drive'da kalır.

In [ ]:
from google.colab import runtime
print(f"Çalışma zamanı kapatılıyor: {datetime.now():%H:%M}")
runtime.unassign()